# Elliott and Sparrow 2012 Beam Launcher

Use this notebook to reproduce the paper-style flexible-beam waveforms and, when a Digifly Phase 2 run is available, display the recorded soma voltage traces for the GF/TTMn escape cells. No terminal commands are required.

## Setup

This cell finds the app whether the notebook is opened from the standalone project or from `Digifly Public/Phase 2/apps/Elliott_Sparrow_2012_Beam`. It also reuses Phase 2's existing `records.csv` voltage-trace conventions.

In [ ]:
from pathlib import Path
import importlib
import json
import re
import sys
from typing import Iterable

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


ESCAPE_CELL_IDS = [10000, 10002, 10068, 10110]
MOTOR_CELL_IDS = [10068, 10110]
GF_CELL_IDS = [10000, 10002]
CELL_LABELS = {
    10000: "GF",
    10002: "GF",
    10068: "TTMn",
    10110: "TTMn",
    11446: "PSI",
    11654: "PSI",
}


def find_app_root() -> Path:
    marker = Path("tools") / "beam_waveform_model.py"
    starts = [Path.cwd(), *Path.cwd().parents]
    known = [
        Path("/Users/juanlopez2016/Desktop/Digifly Public/Phase 2/apps/Elliott_Sparrow_2012_Beam"),
        Path("/Users/juanlopez2016/Desktop/Elliott_Sparrow_2012_Beam"),
    ]
    for base in [*starts, *known]:
        try:
            base = base.expanduser().resolve()
        except Exception:
            continue
        if (base / marker).exists():
            return base
        nested = base / "Phase 2" / "apps" / "Elliott_Sparrow_2012_Beam"
        if (nested / marker).exists():
            return nested.resolve()
    raise RuntimeError("Could not find tools/beam_waveform_model.py. Open this notebook from the Elliott_Sparrow_2012_Beam app folder.")


def find_phase2_root(app_root: Path) -> Path | None:
    if app_root.parts[-3:] == ("Phase 2", "apps", "Elliott_Sparrow_2012_Beam"):
        return app_root.parents[1]
    for parent in [app_root, *app_root.parents]:
        if parent.name == "Phase 2" and (parent / "digifly").exists():
            return parent
    known = Path("/Users/juanlopez2016/Desktop/Digifly Public/Phase 2")
    return known.resolve() if (known / "digifly").exists() else None


APP_ROOT = find_app_root()
PHASE2_ROOT = find_phase2_root(APP_ROOT)
DIGIFLY_ROOT = PHASE2_ROOT.parent if PHASE2_ROOT is not None else None
TOOLS_DIR = APP_ROOT / "tools"

for candidate in [TOOLS_DIR, PHASE2_ROOT]:
    if candidate is not None and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

def refresh_beam_model():
    global _beam_waveform_model, BeamParams, generate_condition, write_outputs
    if str(TOOLS_DIR) not in sys.path:
        sys.path.insert(0, str(TOOLS_DIR))
    expected_path = (TOOLS_DIR / "beam_waveform_model.py").resolve()
    existing = sys.modules.get("beam_waveform_model")
    if existing is not None:
        try:
            existing_path = Path(getattr(existing, "__file__", "")).expanduser().resolve()
        except Exception:
            existing_path = None
        if existing_path != expected_path:
            sys.modules.pop("beam_waveform_model", None)
    module = importlib.import_module("beam_waveform_model")
    _beam_waveform_model = importlib.reload(module)
    BeamParams = _beam_waveform_model.BeamParams
    generate_condition = _beam_waveform_model.generate_condition
    write_outputs = _beam_waveform_model.write_outputs
    return BeamParams, generate_condition, write_outputs


BeamParams, generate_condition, write_outputs = refresh_beam_model()

# Phase 2 imports are intentionally lazy. Some NEURON/display stacks can fail
# during import in headless kernels, but the beam-only workflow should still open.
PHASE2_IMPORT_ERROR = None

DEFAULT_OUTPUT_ROOT = (
    PHASE2_ROOT / "workbench_runs" / "elliott_sparrow_beam"
    if PHASE2_ROOT is not None
    else APP_ROOT / "outputs"
)

DEFAULT_PHASE2_RUNS_ROOT = (
    DIGIFLY_ROOT / "Phase 1" / "manc_v1.2.1" / "export_swc" / "hemi_runs"
    if DIGIFLY_ROOT is not None
    else APP_ROOT / "runs"
)

CONDITIONS = [
    "wildtype_one_leg",
    "wildtype_jump",
    "phase2_gated_jump",
    "phase2_gated_shakB2",
    "standing_still",
    "cs_jump",
    "shakB2_one_leg",
    "shakB2_six_leg",
    "amph26_jump",
    "walking",
    "adhesion_grip",
    "flight_downdraft",
    "larval_wildtype",
    "larval_parkin25",
]

DEFAULT_DURATIONS_MS = {
    "standing_still": 1000.0,
    "walking": 1200.0,
    "adhesion_grip": 600.0,
    "flight_downdraft": 1200.0,
    "larval_wildtype": 20000.0,
    "larval_parkin25": 20000.0,
}

DEFAULT_DT_MS = {
    "standing_still": 0.5,
    "larval_wildtype": 2.0,
    "larval_parkin25": 2.0,
}

display(Markdown(
    f"**App root:** `{APP_ROOT}`  \n"
    f"**Digifly Phase 2 root:** `{PHASE2_ROOT if PHASE2_ROOT else 'not found'}`  \n"
    f"**Default beam output root:** `{DEFAULT_OUTPUT_ROOT}`  \n"
    f"**Default Phase 2 run root:** `{DEFAULT_PHASE2_RUNS_ROOT}`"
))

if PHASE2_IMPORT_ERROR is not None:
    display(Markdown(f"Phase 2 simulation launch is unavailable in this kernel: `{PHASE2_IMPORT_ERROR}`"))

## Shared Plot Helpers

The voltage helper follows Phase 2's existing convention: a run folder contains `records.csv`, with a time column such as `t_ms` and soma-voltage columns like `10000_soma_v`. The correlation helper treats TTMn spikes as the neural event that drives the modeled muscle impulse and beam response. For `phase2_gated_jump` and `phase2_gated_shakB2`, no TTMn spike means no modeled beam jump.

In [ ]:
def default_duration_ms(condition: str) -> float:
    return DEFAULT_DURATIONS_MS.get(condition, 80.0)


def default_dt_ms(condition: str) -> float:
    return DEFAULT_DT_MS.get(condition, 0.025)


def simulation_gated_condition(condition: str) -> bool:
    return str(condition).strip().lower() in {
        "phase2_gated_jump",
        "simulation_gated_jump",
        "digifly_gated_jump",
        "phase2_escape_jump",
        "phase2_gated_shakb2",
        "phase2_gated_shak-b2",
        "simulation_gated_shakb2",
        "phase2_shakb2_jump",
        "phase2_shak-b2_jump",
    }


def find_time_column(records: pd.DataFrame) -> str:
    for name in ("t_ms", "time_ms", "time", "t"):
        if name in records.columns:
            return str(name)
    raise ValueError(f"records.csv has no recognizable time column. Columns: {list(records.columns)[:12]}")


def trace_column(records: pd.DataFrame, neuron_id: int) -> str | None:
    preferred = f"{int(neuron_id)}_soma_v"
    if preferred in records.columns:
        return preferred
    prefix = f"{int(neuron_id)}"
    for column in records.columns:
        name = str(column)
        if name.startswith(prefix) and name.endswith("_soma_v"):
            return name
    return None


def recorded_neuron_ids(run_dir: str | Path) -> list[int]:
    run_dir = Path(run_dir).expanduser().resolve()
    try:
        from digifly.phase2.workbench.browser_visualizer import recorded_neuron_ids as phase2_recorded_neuron_ids
        ids = phase2_recorded_neuron_ids(run_dir)
        if ids:
            return [int(x) for x in ids]
    except BaseException:
        pass
    records_path = run_dir / "records.csv"
    if not records_path.exists():
        return []
    header = records_path.open("r", encoding="utf-8").readline().strip().split(",")
    out = []
    for column in header:
        if not column.endswith("_soma_v"):
            continue
        match = re.match(r"^(\d+)", column)
        if match:
            out.append(int(match.group(1)))
    return out


def parse_neuron_ids(text: str | Iterable[int] | None, fallback: Iterable[int] = ESCAPE_CELL_IDS) -> list[int]:
    fallback_ids = [int(x) for x in fallback]
    if text is None:
        return fallback_ids
    if isinstance(text, str):
        cleaned = text.strip().lower()
        if cleaned in {"", "all", "all cells", "*"}:
            return fallback_ids
        vals = re.findall(r"\d+", text)
        return [int(x) for x in vals] if vals else fallback_ids
    return [int(x) for x in text]


def load_spike_events(
    run_dir: str | Path | None,
    neuron_ids: str | Iterable[int] | None = None,
    threshold_mV: float = 0.0,
) -> pd.DataFrame:
    """Return spike events from Phase 2 output, preferring spike_times.csv.

    Columns: neuron_id, spike_time_ms, role, source.
    If spike_times.csv/spikes.csv is absent, derive threshold crossings from records.csv.
    """
    columns = ["neuron_id", "spike_time_ms", "role", "source"]
    if run_dir in (None, ""):
        return pd.DataFrame(columns=columns)
    run_dir = Path(run_dir).expanduser().resolve()
    fallback_ids = recorded_neuron_ids(run_dir) or ESCAPE_CELL_IDS
    requested_ids = set(parse_neuron_ids(neuron_ids, fallback=fallback_ids))

    for name in ("spike_times.csv", "spikes.csv"):
        path = run_dir / name
        if not path.exists():
            continue
        spikes = pd.read_csv(path)
        id_col = next((c for c in ("neuron_id", "bodyId", "bodyid", "id") if c in spikes.columns), None)
        time_col = next((c for c in ("spike_time_ms", "t_ms", "time_ms", "time", "t") if c in spikes.columns), None)
        if id_col is None or time_col is None:
            continue
        out = spikes[[id_col, time_col]].copy()
        out.columns = ["neuron_id", "spike_time_ms"]
        out["neuron_id"] = pd.to_numeric(out["neuron_id"], errors="coerce")
        out["spike_time_ms"] = pd.to_numeric(out["spike_time_ms"], errors="coerce")
        out = out.dropna().copy()
        out["neuron_id"] = out["neuron_id"].astype(int)
        out = out[out["neuron_id"].isin(requested_ids)].sort_values(["spike_time_ms", "neuron_id"])
        out["role"] = out["neuron_id"].map(CELL_LABELS).fillna("cell")
        out["source"] = name
        return out[columns].reset_index(drop=True)

    records_path = run_dir / "records.csv"
    if not records_path.exists():
        return pd.DataFrame(columns=columns)

    records = pd.read_csv(records_path)
    time_col = find_time_column(records)
    t = pd.to_numeric(records[time_col], errors="coerce").to_numpy(dtype=float)
    events = []
    for nid in sorted(requested_ids):
        col = trace_column(records, nid)
        if col is None:
            continue
        v = pd.to_numeric(records[col], errors="coerce").to_numpy(dtype=float)
        valid = np.isfinite(t) & np.isfinite(v)
        tt = t[valid]
        vv = v[valid]
        if tt.size < 2:
            continue
        crossings = np.flatnonzero((vv[:-1] < float(threshold_mV)) & (vv[1:] >= float(threshold_mV))) + 1
        for idx in crossings:
            events.append({
                "neuron_id": int(nid),
                "spike_time_ms": float(tt[idx]),
                "role": CELL_LABELS.get(int(nid), "cell"),
                "source": f"records.csv threshold >= {threshold_mV:g} mV",
            })
    return pd.DataFrame(events, columns=columns).sort_values(["spike_time_ms", "neuron_id"]).reset_index(drop=True)


def display_activity_beam_correlation(
    beam_df: pd.DataFrame,
    condition: str,
    phase2_run: str | Path | None = None,
    spike_events: pd.DataFrame | None = None,
    stim_time_ms: float = 20.0,
    motor_ids: Iterable[int] = MOTOR_CELL_IDS,
) -> pd.DataFrame:
    """Show the timing bridge from neural events to the beam response."""
    if spike_events is None:
        spike_events = load_spike_events(phase2_run, neuron_ids=ESCAPE_CELL_IDS)
    motor_ids = {int(x) for x in motor_ids}
    peak_idx = beam_df["vector"].astype(float).idxmax()
    peak_time = float(beam_df.loc[peak_idx, "t_ms"])
    peak_vector = float(beam_df.loc[peak_idx, "vector"])
    peak_vertical = float(beam_df.loc[peak_idx, "vertical"])
    jump_decision = str(beam_df.attrs.get("jump_decision") or "")
    gate_source = str(beam_df.attrs.get("gate_source") or "")
    phase2_motor_spike_count = int(beam_df.attrs.get("phase2_motor_spike_count") or 0)
    sim_gate_closed = jump_decision == "no_jump" and gate_source.startswith("phase2")

    rows = []
    if spike_events is not None and not spike_events.empty:
        gf = spike_events[spike_events["neuron_id"].isin(GF_CELL_IDS)]
        motor = spike_events[spike_events["neuron_id"].isin(motor_ids)]
        if not gf.empty:
            first = gf.iloc[0]
            rows.append({"event": "first GF spike", "time_ms": float(first.spike_time_ms), "neuron_id": int(first.neuron_id), "delta_to_beam_peak_ms": peak_time - float(first.spike_time_ms)})
        if not motor.empty:
            first = motor.iloc[0]
            rows.append({"event": "first TTMn spike driving beam", "time_ms": float(first.spike_time_ms), "neuron_id": int(first.neuron_id), "delta_to_beam_peak_ms": peak_time - float(first.spike_time_ms)})
        else:
            modeled_motor_time = float(stim_time_ms) + 2.0
            rows.append({"event": "no recorded TTMn spike found", "time_ms": np.nan, "neuron_id": np.nan, "delta_to_beam_peak_ms": np.nan})
            if sim_gate_closed:
                rows.append({"event": "beam gate closed: no TTMn spike", "time_ms": np.nan, "neuron_id": np.nan, "delta_to_beam_peak_ms": np.nan})
            else:
                rows.append({"event": "preset motor event used for beam", "time_ms": modeled_motor_time, "neuron_id": np.nan, "delta_to_beam_peak_ms": peak_time - modeled_motor_time})
    else:
        modeled_motor_time = float(stim_time_ms) + 2.0
        if sim_gate_closed or jump_decision == "no_jump":
            rows.append({"event": "beam decision: no jump", "time_ms": np.nan, "neuron_id": np.nan, "delta_to_beam_peak_ms": np.nan})
        else:
            rows.append({"event": "modeled TTMn/muscle event", "time_ms": modeled_motor_time, "neuron_id": np.nan, "delta_to_beam_peak_ms": peak_time - modeled_motor_time})

    rows.append({"event": "beam vector peak", "time_ms": peak_time, "neuron_id": np.nan, "delta_to_beam_peak_ms": 0.0})
    summary = pd.DataFrame(rows)
    if sim_gate_closed:
        bridge_text = "**Activity-to-beam timing.** This is a simulation-gated condition: Phase 2 produced no TTMn spike, so the beam gate stayed closed and the trace remains flat."
    elif gate_source.startswith("phase2"):
        bridge_text = "**Activity-to-beam timing.** This is a simulation-gated condition: the first Phase 2 TTMn spike drives the modeled muscle/sensor transform."
    else:
        bridge_text = (
            "**Activity-to-beam timing.** The voltage trace contributes spike timing; "
            "the beam trace is a muscle/sensor response driven by TTMn spike time plus the model's muscle/beam transform."
        )
    display(Markdown(bridge_text))
    display(summary)
    display(pd.Series({
        "condition": condition,
        "beam_peak_time_ms": peak_time,
        "beam_peak_vector": peak_vector,
        "beam_vertical_at_peak": peak_vertical,
        "jump_decision": jump_decision or None,
        "gate_source": gate_source or None,
        "phase2_motor_spike_count": phase2_motor_spike_count,
    }).to_frame("value"))
    if spike_events is not None and not spike_events.empty:
        display(Markdown("Detected neural spike events used for alignment:"))
        display(spike_events.head(40))
    return summary


def plot_voltage_traces(
    run_dir: str | Path,
    neuron_ids: str | Iterable[int] | None = None,
    max_traces: int = 12,
    spike_events: pd.DataFrame | None = None,
) -> pd.DataFrame | None:
    run_dir = Path(run_dir).expanduser().resolve()
    records_path = run_dir / "records.csv"
    if not records_path.exists():
        display(Markdown(f"No `records.csv` found at `{records_path}`."))
        return None

    records = pd.read_csv(records_path)
    time_col = find_time_column(records)
    requested_ids = parse_neuron_ids(neuron_ids, fallback=recorded_neuron_ids(run_dir) or ESCAPE_CELL_IDS)
    if spike_events is None:
        spike_events = load_spike_events(run_dir, requested_ids)
    trace_pairs = []
    missing = []
    for nid in requested_ids:
        col = trace_column(records, nid)
        if col is None:
            missing.append(nid)
        else:
            trace_pairs.append((nid, col))

    if not trace_pairs:
        display(Markdown(f"No requested soma voltage traces were found in `{records_path}`."))
        available = recorded_neuron_ids(run_dir)
        if available:
            display(Markdown(f"Recorded neuron IDs: `{available}`"))
        return records

    try:
        import matplotlib.pyplot as plt
    except Exception as exc:
        display(Markdown(f"Matplotlib is unavailable, so showing the voltage table head. Import error: `{exc}`"))
        display(records[[time_col] + [col for _, col in trace_pairs]].head())
        return records

    plot_pairs = trace_pairs[: int(max_traces)]
    fig, ax = plt.subplots(figsize=(11, 4.8))
    t = records[time_col].to_numpy(dtype=float)
    for nid, col in plot_pairs:
        label = f"{nid} {CELL_LABELS.get(nid, '')}".strip()
        v = records[col].to_numpy(dtype=float)
        line = ax.plot(t, v, linewidth=1.35, label=label)[0]
        if spike_events is not None and not spike_events.empty:
            st = spike_events.loc[spike_events["neuron_id"].astype(int) == int(nid), "spike_time_ms"].to_numpy(dtype=float)
            st = st[np.isfinite(st)]
            if st.size:
                ax.scatter(st, np.interp(st, t, v), s=26, color=line.get_color(), edgecolor="white", linewidth=0.5, zorder=4)
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Soma voltage (mV)")
    ax.set_title(f"Phase 2 soma voltage traces: {run_dir.name}")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best", ncol=2)
    if len(trace_pairs) > len(plot_pairs):
        ax.text(0.01, 0.98, f"Showing {len(plot_pairs)} of {len(trace_pairs)} traces", transform=ax.transAxes, va="top", ha="left", fontsize=9)
    fig.tight_layout()
    plt.show()

    if missing:
        display(Markdown(f"Missing requested voltage traces: `{missing}`"))
    return records


def plot_beam_waveform(
    df: pd.DataFrame,
    condition: str,
    spike_events: pd.DataFrame | None = None,
    motor_ids: Iterable[int] = MOTOR_CELL_IDS,
) -> None:
    try:
        import matplotlib.pyplot as plt
    except Exception as exc:
        display(Markdown(f"Matplotlib is unavailable, so showing the first rows instead. Import error: `{exc}`"))
        display(df.head())
        return

    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    axes[0].plot(df["t_ms"], df["vertical"], label="vertical", linewidth=1.4)
    axes[0].plot(df["t_ms"], df["horizontal"], label="horizontal", linewidth=1.1, alpha=0.85)
    axes[0].set_ylabel("Beam axis signal")
    axes[0].legend(loc="best")
    axes[0].grid(alpha=0.25)

    axes[1].plot(df["t_ms"], df["vector"], label="vector", linewidth=1.4)
    axes[1].set_xlabel("Time (ms)")
    axes[1].set_ylabel("Beam vector")
    axes[1].legend(loc="best")
    axes[1].grid(alpha=0.25)

    if spike_events is not None and not spike_events.empty:
        motor_ids = {int(x) for x in motor_ids}
        for _, row in spike_events.iterrows():
            nid = int(row["neuron_id"])
            st = float(row["spike_time_ms"])
            if nid in motor_ids:
                color, alpha, style = "#d62728", 0.45, "--"
                label = "TTMn spike"
            elif nid in GF_CELL_IDS:
                color, alpha, style = "#555555", 0.25, ":"
                label = "GF spike"
            else:
                continue
            for ax in axes:
                ax.axvline(st, color=color, alpha=alpha, linestyle=style, linewidth=1.0)
        axes[0].text(0.01, 0.98, "dotted=GF spikes, dashed=TTMn spikes", transform=axes[0].transAxes, va="top", ha="left", fontsize=9)

    fig.suptitle(f"Beam waveform: {condition}")
    fig.tight_layout()
    plt.show()


def plot_activity_locked_view(
    run_dir: str | Path,
    beam_df: pd.DataFrame,
    condition: str,
    neuron_ids: str | Iterable[int] | None = None,
    spike_events: pd.DataFrame | None = None,
    motor_ids: Iterable[int] = MOTOR_CELL_IDS,
    focus_pad_ms: float = 20.0,
) -> pd.DataFrame | None:
    """Plot voltage and beam traces on one shared time axis."""
    run_dir = Path(run_dir).expanduser().resolve()
    records_path = run_dir / "records.csv"
    if not records_path.exists():
        display(Markdown(f"No `records.csv` found at `{records_path}`, so the time-locked voltage/beam view cannot be drawn."))
        return None

    records = pd.read_csv(records_path)
    time_col = find_time_column(records)
    requested_ids = parse_neuron_ids(neuron_ids, fallback=recorded_neuron_ids(run_dir) or ESCAPE_CELL_IDS)
    if spike_events is None:
        spike_events = load_spike_events(run_dir, requested_ids)

    trace_pairs = []
    for nid in requested_ids:
        col = trace_column(records, nid)
        if col is not None:
            trace_pairs.append((int(nid), col))
    if not trace_pairs:
        display(Markdown(f"No selected soma voltage traces were found in `{records_path}`."))
        return records

    try:
        import matplotlib.pyplot as plt
    except Exception as exc:
        display(Markdown(f"Matplotlib is unavailable, so the time-locked view cannot be drawn. Import error: `{exc}`"))
        return records

    record_t = pd.to_numeric(records[time_col], errors="coerce").to_numpy(dtype=float)
    beam_t = pd.to_numeric(beam_df["t_ms"], errors="coerce").to_numpy(dtype=float)
    event_times = []
    if spike_events is not None and not spike_events.empty:
        event_times = pd.to_numeric(spike_events["spike_time_ms"], errors="coerce").dropna().astype(float).tolist()
    peak_t = float(beam_df.loc[beam_df["vector"].astype(float).idxmax(), "t_ms"])
    anchors = [peak_t, *event_times]
    anchors = [x for x in anchors if np.isfinite(x)]
    if anchors:
        x_min = max(float(np.nanmin([np.nanmin(record_t), np.nanmin(beam_t)])), min(anchors) - float(focus_pad_ms))
        x_max = min(float(np.nanmax([np.nanmax(record_t), np.nanmax(beam_t)])), max(anchors) + max(50.0, float(focus_pad_ms)))
        if x_max <= x_min:
            x_min = float(np.nanmin(beam_t))
            x_max = float(np.nanmax(beam_t))
    else:
        x_min = float(np.nanmin(beam_t))
        x_max = float(np.nanmax(beam_t))

    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True, gridspec_kw={"height_ratios": [1.4, 1.0, 1.0]})
    ax_v, ax_beam, ax_vec = axes

    for nid, col in trace_pairs:
        vals = pd.to_numeric(records[col], errors="coerce").to_numpy(dtype=float)
        label = f"{nid} {CELL_LABELS.get(nid, '')}".strip()
        line = ax_v.plot(record_t, vals, linewidth=1.25, label=label)[0]
        if spike_events is not None and not spike_events.empty:
            st = spike_events.loc[spike_events["neuron_id"].astype(int) == int(nid), "spike_time_ms"].to_numpy(dtype=float)
            st = st[np.isfinite(st)]
            if st.size:
                ax_v.scatter(st, np.interp(st, record_t, vals), s=28, color=line.get_color(), edgecolor="white", linewidth=0.5, zorder=4)

    ax_beam.plot(beam_t, beam_df["vertical"], label="beam vertical", linewidth=1.35)
    ax_beam.plot(beam_t, beam_df["horizontal"], label="beam horizontal", linewidth=1.1, alpha=0.85)
    ax_vec.plot(beam_t, beam_df["vector"], label="beam vector", linewidth=1.45, color="#2ca02c")
    ax_vec.scatter([peak_t], [float(beam_df.loc[beam_df["vector"].astype(float).idxmax(), "vector"])], s=42, color="#2ca02c", edgecolor="white", linewidth=0.6, zorder=5, label="beam peak")

    if spike_events is not None and not spike_events.empty:
        motor_ids = {int(x) for x in motor_ids}
        seen_labels = set()
        for _, row in spike_events.iterrows():
            nid = int(row["neuron_id"])
            st = float(row["spike_time_ms"])
            if not np.isfinite(st):
                continue
            if nid in motor_ids:
                color, alpha, style, label = "#d62728", 0.55, "--", "TTMn spike"
            elif nid in GF_CELL_IDS:
                color, alpha, style, label = "#555555", 0.35, ":", "GF spike"
            else:
                color, alpha, style, label = "#9467bd", 0.25, ":", "other spike"
            draw_label = label if label not in seen_labels else None
            seen_labels.add(label)
            for ax in axes:
                ax.axvline(st, color=color, alpha=alpha, linestyle=style, linewidth=1.1, label=draw_label if ax is ax_vec else None)

    jump_decision = str(beam_df.attrs.get("jump_decision") or "")
    gate_source = str(beam_df.attrs.get("gate_source") or "")
    if jump_decision == "no_jump":
        note = "no beam jump"
        if gate_source.startswith("phase2"):
            note = "no beam jump: no Phase 2 TTMn spike"
        ax_vec.text(0.01, 0.92, note, transform=ax_vec.transAxes, va="top", ha="left", fontsize=10)

    ax_v.set_ylabel("Soma V (mV)")
    ax_beam.set_ylabel("Beam axes")
    ax_vec.set_ylabel("Beam vector")
    ax_vec.set_xlabel("Time (ms)")
    ax_v.set_title(f"Time-locked neural activity to beam response: {condition}")
    for ax in axes:
        ax.grid(True, alpha=0.25)
        ax.set_xlim(x_min, x_max)
    ax_v.legend(loc="best", ncol=2)
    ax_beam.legend(loc="best", ncol=2)
    ax_vec.legend(loc="best", ncol=3)
    fig.tight_layout()
    plt.show()
    return records


## Beam Waveform Launcher

Choose a paper condition and generate the flexible-beam waveform. If you paste a Phase 2 run folder, the notebook reads `spike_times.csv` or derives threshold crossings from `records.csv`, then draws a single time-locked plot: soma voltages on top, beam axes in the middle, and beam vector on the bottom. GF and TTMn spike markers run vertically through all panels.

Use `phase2_gated_jump` when the Phase 2 TTMn spike should decide whether the beam jumps. Use `phase2_gated_shakB2` with a no-gap Phase 2 run when you want the shakB2/no-gap condition to fail by losing the TTMn spike rather than by forcing the beam trace flat afterward.

In [ ]:
def current_beam_api():
    if "refresh_beam_model" in globals():
        return refresh_beam_model()
    import importlib as _importlib
    import beam_waveform_model as _beam_waveform_model_fallback
    _beam_waveform_model_fallback = _importlib.reload(_beam_waveform_model_fallback)
    return (
        _beam_waveform_model_fallback.BeamParams,
        _beam_waveform_model_fallback.generate_condition,
        _beam_waveform_model_fallback.write_outputs,
    )


def run_waveform(
    condition: str = "wildtype_jump",
    output_root: str | Path = DEFAULT_OUTPUT_ROOT,
    phase2_run: str | Path | None = None,
    voltage_ids: str | Iterable[int] | None = "all",
    duration_ms: float | None = None,
    dt_ms: float | None = None,
    stim_time_ms: float = 20.0,
    seed: int = 7,
    show_beam_plot: bool = True,
    show_voltage_plot: bool = True,
    show_correlation: bool = True,
) -> pd.DataFrame:
    condition = str(condition).strip()
    phase2_gated = simulation_gated_condition(condition)
    out_root = Path(output_root).expanduser().resolve()
    out_dir = out_root / condition
    phase2_path = None
    spike_events = pd.DataFrame(columns=["neuron_id", "spike_time_ms", "role", "source"])
    gate_spike_events = pd.DataFrame(columns=["neuron_id", "spike_time_ms", "role", "source"])
    if phase2_run not in (None, ""):
        phase2_path = Path(phase2_run).expanduser().resolve()
        spike_events = load_spike_events(phase2_path, neuron_ids=voltage_ids)
        gate_spike_events = load_spike_events(phase2_path, neuron_ids=MOTOR_CELL_IDS)
        if not gate_spike_events.empty:
            spike_events = pd.concat([spike_events, gate_spike_events], ignore_index=True)
            spike_events = spike_events.drop_duplicates(["neuron_id", "spike_time_ms", "source"]).sort_values(["spike_time_ms", "neuron_id"]).reset_index(drop=True)
        if not (phase2_path / "spike_times.csv").exists():
            if phase2_gated:
                display(Markdown(f"No `spike_times.csv` found in `{phase2_path}`. This simulation-gated beam condition will stay flat unless a TTMn spike is available in `spike_times.csv`."))
            else:
                display(Markdown(f"No `spike_times.csv` found in `{phase2_path}`. The beam trace will use the condition preset, but voltage spike markers can still be derived from `records.csv`."))

    motor_spikes = gate_spike_events[gate_spike_events["neuron_id"].astype(int).isin(MOTOR_CELL_IDS)] if not gate_spike_events.empty else gate_spike_events
    if phase2_gated:
        phase2_for_beam = phase2_path
    else:
        phase2_for_beam = phase2_path if phase2_path and (phase2_path / "spike_times.csv").exists() and not motor_spikes.empty else None
    if phase2_path is not None and motor_spikes.empty:
        if phase2_gated:
            display(Markdown("No TTMn spikes (`10068` or `10110`) were detected, so the simulation-gated beam condition will show no jump."))
        elif (phase2_path / "spike_times.csv").exists():
            display(Markdown("`spike_times.csv` has no TTMn spikes (`10068` or `10110`), so the beam trace uses the condition preset while voltage traces still show the recorded cells."))

    auto_duration_ms = default_duration_ms(condition)
    if duration_ms is None and phase2_for_beam is not None and not motor_spikes.empty:
        latest_spike = float(pd.to_numeric(motor_spikes["spike_time_ms"], errors="coerce").dropna().max())
        if np.isfinite(latest_spike):
            auto_duration_ms = max(auto_duration_ms, latest_spike + 60.0)

    BeamParamsLive, generate_condition_live, write_outputs_live = current_beam_api()
    params = BeamParamsLive(
        duration_ms=float(duration_ms if duration_ms is not None else auto_duration_ms),
        dt_ms=float(dt_ms if dt_ms is not None else default_dt_ms(condition)),
        stim_time_ms=float(stim_time_ms),
        seed=int(seed),
    )
    df = generate_condition_live(condition, params, phase2_run=phase2_for_beam)
    write_outputs_live(df, out_dir, condition, params, phase2_for_beam)

    summary_path = out_dir / f"{condition}_summary.json"
    summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
    display(Markdown(f"**Wrote beam outputs to:** `{out_dir}`"))
    display(pd.Series({
        "condition": condition,
        "rows": int(df.shape[0]),
        "t_stop_ms": float(df["t_ms"].max()),
        "vertical_peak_abs": float(df["vertical"].abs().max()),
        "horizontal_peak_abs": float(df["horizontal"].abs().max()),
        "vector_peak": float(df["vector"].max()),
        "response_events": summary.get("response_events"),
        "jump_decision": summary.get("jump_decision"),
        "gate_source": summary.get("gate_source"),
        "phase2_motor_spike_count": summary.get("phase2_motor_spike_count"),
    }).to_frame("value"))

    if show_correlation:
        display_activity_beam_correlation(
            df,
            condition=condition,
            phase2_run=phase2_path,
            spike_events=spike_events,
            stim_time_ms=stim_time_ms,
        )
    if show_voltage_plot and phase2_path is not None and (phase2_path / "records.csv").exists():
        plot_activity_locked_view(phase2_path, df, condition, neuron_ids=voltage_ids, spike_events=spike_events)
    elif show_beam_plot:
        plot_beam_waveform(df, condition, spike_events=spike_events)
    return df


try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    condition_widget = widgets.Dropdown(options=CONDITIONS, value="phase2_gated_jump", description="Condition")
    output_widget = widgets.Text(value=str(DEFAULT_OUTPUT_ROOT), description="Output root", layout=widgets.Layout(width="95%"))
    phase2_widget = widgets.Text(value="", description="Phase 2 run", layout=widgets.Layout(width="95%"))
    voltage_ids_widget = widgets.Text(value="all", description="Voltage IDs", layout=widgets.Layout(width="95%"))
    duration_widget = widgets.FloatText(value=0.0, description="Duration ms")
    dt_widget = widgets.FloatText(value=0.0, description="dt ms")
    stim_widget = widgets.FloatText(value=20.0, description="Stim ms")
    seed_widget = widgets.IntText(value=7, description="Seed")
    button = widgets.Button(description="Generate beam + activity link", button_style="primary")
    output_area = widgets.Output()

    def on_generate(_):
        with output_area:
            clear_output(wait=True)
            duration = None if duration_widget.value <= 0 else duration_widget.value
            dt = None if dt_widget.value <= 0 else dt_widget.value
            run_waveform(
                condition=condition_widget.value,
                output_root=output_widget.value,
                phase2_run=phase2_widget.value.strip() or None,
                voltage_ids=voltage_ids_widget.value,
                duration_ms=duration,
                dt_ms=dt,
                stim_time_ms=stim_widget.value,
                seed=seed_widget.value,
            )

    button.on_click(on_generate)
    display(widgets.VBox([
        condition_widget,
        output_widget,
        phase2_widget,
        voltage_ids_widget,
        widgets.HBox([duration_widget, dt_widget, stim_widget, seed_widget]),
        button,
        output_area,
    ]))
except Exception as exc:
    display(Markdown(f"Interactive widgets are unavailable: `{exc}`. Edit the values below and run the cell."))
    df = run_waveform(condition="phase2_gated_jump", output_root=DEFAULT_OUTPUT_ROOT, phase2_run=None)

## Plot Voltages From An Existing Phase 2 Run

Use this when you already have a Digifly run folder. Paste the folder that contains `records.csv`; the plot will use the same voltage-column conventions as the Phase 2 browser visualizer.

In [ ]:
PHASE2_RUN_DIR = ""  # Example: "/Users/juanlopez2016/Desktop/Digifly Public/Phase 1/manc_v1.2.1/export_swc/hemi_runs/single_neuron_debug"
VOLTAGE_NEURON_IDS = "all"

if PHASE2_RUN_DIR:
    records_df = plot_voltage_traces(PHASE2_RUN_DIR, neuron_ids=VOLTAGE_NEURON_IDS)
else:
    display(Markdown("Paste a Phase 2 run folder into `PHASE2_RUN_DIR` and rerun this cell to plot soma voltages."))

## Optional: Launch The Digifly Escape Circuit Run

This cell runs the Digifly Phase 2 escape-circuit template from the notebook, then plots soma voltages and generates the matching beam waveform. The `wildtype_gap` preset enables GF-to-TTMn electrical coupling; the `shakB2_no_gap` preset disables those gap junctions and uses the simulation-gated shakB2 beam condition. Tune the current clamp, synaptic weights, and gap conductance until the wildtype run makes TTMn spike and the no-gap run does not.

Leave `RUN_DIGIFLY_ESCAPE = False` until you are ready to run NEURON locally.

In [ ]:
def _resolve_digifly_path(value: str | Path | None) -> str | None:
    if value in (None, ""):
        return None
    p = Path(value)
    if not p.is_absolute() and DIGIFLY_ROOT is not None:
        p = DIGIFLY_ROOT / p
    return str(p.expanduser().resolve())


def _escape_gap_pairs(include_psi_gap_pairs: bool = False) -> list[tuple[int, int]]:
    pairs = [(10000, 10110), (10002, 10068)]
    if include_psi_gap_pairs:
        pairs.extend([(10000, 11446), (10000, 11654), (10002, 11446), (10002, 11654)])
    return pairs


def build_escape_gap_config(
    enabled: bool = True,
    mode: str = "rectifying",
    rectify_direction: str = "a_to_b",
    g_uS: float = 0.001,
    default_site: str = "ais",
    all_synapses: bool = True,
    max_synapses: int = 1,
    mechanisms_dir: str | Path | None = None,
    include_psi_gap_pairs: bool = False,
    gap_pairs: Iterable[tuple[int, int]] | None = None,
) -> dict:
    mechanisms_dir = mechanisms_dir or (PHASE2_ROOT / "data" if PHASE2_ROOT is not None else None)
    base = {
        "enabled": bool(enabled),
        "mechanisms_dir": _resolve_digifly_path(mechanisms_dir),
        "default_site": str(default_site),
        "default_g_uS": float(g_uS),
        "pairs": [],
    }
    if not enabled:
        return base

    mode_norm = str(mode).strip().lower()
    if mode_norm not in {"ohmic", "rectifying"}:
        raise ValueError("gap mode must be 'ohmic' or 'rectifying'")

    cfg_pairs = []
    for a_id, b_id in list(gap_pairs or _escape_gap_pairs(include_psi_gap_pairs=include_psi_gap_pairs)):
        if mode_norm == "ohmic":
            entry = {
                "mode": "ohmic",
                "a_id": int(a_id),
                "b_id": int(b_id),
                "g_uS": float(g_uS),
                "site_a": str(default_site),
                "site_b": str(default_site),
            }
        else:
            entry = {
                "mode": "rectifying",
                "a_id": int(a_id),
                "b_id": int(b_id),
                "direction": str(rectify_direction),
                "g_uS": float(g_uS),
                "site_a": str(default_site),
                "site_b": str(default_site),
            }
        entry["placement"] = "synapse"
        entry["all_synapses"] = bool(all_synapses)
        entry["max_synapses"] = int(max_synapses)
        cfg_pairs.append(entry)
    base["pairs"] = cfg_pairs
    return base


def build_escape_config(
    run_id: str = "elliott_sparrow_2012_escape_wildtype_gap",
    condition: str = "phase2_gated_jump",
    tstop_ms: float = 80.0,
    dt_ms: float = 0.025,
    iclamp_amp_nA: float = 1.0,
    iclamp_delay_ms: float = 20.0,
    iclamp_dur_ms: float = 1.0,
    default_weight_uS: float = 6e-6,
    default_delay_ms: float | None = None,
    syn_tau1_ms: float = 0.5,
    syn_tau2_ms: float = 3.0,
    syn_e_rev_mV: float = 0.0,
    post_active: bool = True,
    gap_enabled: bool = True,
    gap_mode: str = "rectifying",
    gap_rectify_direction: str = "a_to_b",
    gap_g_uS: float = 0.001,
    gap_all_synapses: bool = True,
    gap_max_synapses: int = 1,
    include_psi_gap_pairs: bool = False,
) -> dict:
    template_path = APP_ROOT / "configs" / "digifly_escape_run_template.json"
    cfg = json.loads(template_path.read_text(encoding="utf-8"))
    cfg["run_id"] = str(run_id)
    cfg["tstop_ms"] = float(tstop_ms)
    cfg["dt_ms"] = float(dt_ms)
    cfg["iclamp_amp_nA"] = float(iclamp_amp_nA)
    cfg["iclamp_delay_ms"] = float(iclamp_delay_ms)
    cfg["iclamp_dur_ms"] = float(iclamp_dur_ms)
    cfg["default_weight_uS"] = float(default_weight_uS)
    cfg["syn_tau1_ms"] = float(syn_tau1_ms)
    cfg["syn_tau2_ms"] = float(syn_tau2_ms)
    cfg["syn_e_rev_mV"] = float(syn_e_rev_mV)
    cfg["post_active"] = bool(post_active)
    if default_delay_ms is not None:
        cfg["default_delay_ms"] = float(default_delay_ms)
    cfg["record"] = {"soma_v": "all", "spikes": "all", "spike_thresh_mV": 0.0}

    for key in ("swc_dir", "morph_swc_dir", "runs_root", "edges_root", "master_csv"):
        cfg[key] = _resolve_digifly_path(cfg.get(key))

    neuron_ids = set(int(x) for x in cfg.get("selection", {}).get("neuron_ids", ESCAPE_CELL_IDS))
    if include_psi_gap_pairs:
        neuron_ids.update([11446, 11654])
    cfg["selection"] = {"mode": "custom", "neuron_ids": sorted(neuron_ids)}
    cfg["seeds"] = [10000, 10002]
    cfg["gap"] = build_escape_gap_config(
        enabled=gap_enabled,
        mode=gap_mode,
        rectify_direction=gap_rectify_direction,
        g_uS=gap_g_uS,
        all_synapses=gap_all_synapses,
        max_synapses=gap_max_synapses,
        include_psi_gap_pairs=include_psi_gap_pairs,
    )
    cfg["elliott_sparrow_condition"] = {
        "condition": str(condition),
        "stim_time_ms": float(iclamp_delay_ms),
        "left_ttmn_id": 10068,
        "right_ttmn_id": 10110,
        "beam_gate": "TTMn spike presence in spike_times.csv",
    }
    return cfg


def escape_preset_kwargs(preset: str) -> dict:
    preset = str(preset).strip().lower()
    if preset in {"wildtype", "wildtype_gap", "cs_gap"}:
        return {
            "condition": "phase2_gated_jump",
            "run_id": "elliott_sparrow_2012_escape_wildtype_gap",
            "gap_enabled": True,
        }
    if preset in {"shakb2", "shakb2_no_gap", "shak-b2_no_gap"}:
        return {
            "condition": "phase2_gated_shakB2",
            "run_id": "elliott_sparrow_2012_escape_shakb2_no_gap",
            "gap_enabled": False,
        }
    if preset in {"still", "standing", "fly_stands_still"}:
        return {
            "condition": "standing_still",
            "run_id": "elliott_sparrow_2012_fly_stands_still",
            "run_phase2": False,
        }
    raise ValueError("SIM_PRESET must be 'wildtype_gap', 'shakB2_no_gap', or 'fly_stands_still'.")


def launch_digifly_escape_and_plot(
    condition: str = "phase2_gated_jump",
    run_id: str = "elliott_sparrow_2012_escape_wildtype_gap",
    voltage_ids: str | Iterable[int] = "all",
    tstop_ms: float = 80.0,
    dt_ms: float = 0.025,
    iclamp_amp_nA: float = 1.0,
    iclamp_delay_ms: float = 20.0,
    iclamp_dur_ms: float = 1.0,
    default_weight_uS: float = 6e-6,
    default_delay_ms: float | None = None,
    syn_tau1_ms: float = 0.5,
    syn_tau2_ms: float = 3.0,
    syn_e_rev_mV: float = 0.0,
    post_active: bool = True,
    gap_enabled: bool = True,
    gap_mode: str = "rectifying",
    gap_rectify_direction: str = "a_to_b",
    gap_g_uS: float = 0.001,
    gap_all_synapses: bool = True,
    gap_max_synapses: int = 1,
    include_psi_gap_pairs: bool = False,
):
    if DIGIFLY_ROOT is None:
        raise RuntimeError("Cannot resolve the Digifly repo root from this notebook location.")
    try:
        from digifly.phase2.api import run_walking_simulation
    except BaseException as exc:
        raise RuntimeError(f"Digifly Phase 2 API is unavailable in this kernel: {exc}") from exc

    cfg = build_escape_config(
        run_id=run_id,
        condition=condition,
        tstop_ms=tstop_ms,
        dt_ms=dt_ms,
        iclamp_amp_nA=iclamp_amp_nA,
        iclamp_delay_ms=iclamp_delay_ms,
        iclamp_dur_ms=iclamp_dur_ms,
        default_weight_uS=default_weight_uS,
        default_delay_ms=default_delay_ms,
        syn_tau1_ms=syn_tau1_ms,
        syn_tau2_ms=syn_tau2_ms,
        syn_e_rev_mV=syn_e_rev_mV,
        post_active=post_active,
        gap_enabled=gap_enabled,
        gap_mode=gap_mode,
        gap_rectify_direction=gap_rectify_direction,
        gap_g_uS=gap_g_uS,
        gap_all_synapses=gap_all_synapses,
        gap_max_synapses=gap_max_synapses,
        include_psi_gap_pairs=include_psi_gap_pairs,
    )
    display(Markdown(
        f"Launching Digifly Phase 2 run `{run_id}` with condition `{condition}` "
        f"and gap_enabled=`{gap_enabled}`..."
    ))
    out_dir = Path(run_walking_simulation(cfg)).expanduser().resolve()
    display(Markdown(f"**Digifly run written to:** `{out_dir}`"))

    plot_voltage_traces(out_dir, neuron_ids=voltage_ids)
    beam_df = run_waveform(
        condition=condition,
        output_root=DEFAULT_OUTPUT_ROOT,
        phase2_run=out_dir,
        voltage_ids=voltage_ids,
        duration_ms=tstop_ms,
        dt_ms=dt_ms,
        stim_time_ms=iclamp_delay_ms,
        show_voltage_plot=False,
    )
    return out_dir, beam_df


RUN_DIGIFLY_ESCAPE = False
SIM_PRESET = "wildtype_gap"  # "wildtype_gap", "shakB2_no_gap", or "fly_stands_still"

# Tune these directly in the notebook.
SIM_TSTOP_MS = 80.0
SIM_DT_MS = 0.025
SIM_ICLAMP_AMP_NA = 1.0
SIM_ICLAMP_DELAY_MS = 20.0
SIM_ICLAMP_DUR_MS = 1.0
SIM_DEFAULT_WEIGHT_US = 6e-6
SIM_DEFAULT_DELAY_MS = None
SIM_SYN_TAU1_MS = 0.5
SIM_SYN_TAU2_MS = 3.0
SIM_SYN_E_REV_MV = 0.0
SIM_POST_ACTIVE = True
SIM_GAP_MODE = "rectifying"
SIM_GAP_RECTIFY_DIRECTION = "a_to_b"
SIM_GAP_G_US = 0.001
SIM_GAP_ALL_SYNAPSES = True
SIM_GAP_MAX_SYNAPSES = 1
SIM_INCLUDE_PSI_GAP_PAIRS = False
SIM_VOLTAGE_IDS = "all"

preset = escape_preset_kwargs(SIM_PRESET)
if RUN_DIGIFLY_ESCAPE:
    if preset.get("run_phase2", True):
        RUN_DIR, BEAM_DF = launch_digifly_escape_and_plot(
            condition=preset["condition"],
            run_id=preset["run_id"],
            voltage_ids=SIM_VOLTAGE_IDS,
            tstop_ms=SIM_TSTOP_MS,
            dt_ms=SIM_DT_MS,
            iclamp_amp_nA=SIM_ICLAMP_AMP_NA,
            iclamp_delay_ms=SIM_ICLAMP_DELAY_MS,
            iclamp_dur_ms=SIM_ICLAMP_DUR_MS,
            default_weight_uS=SIM_DEFAULT_WEIGHT_US,
            default_delay_ms=SIM_DEFAULT_DELAY_MS,
            syn_tau1_ms=SIM_SYN_TAU1_MS,
            syn_tau2_ms=SIM_SYN_TAU2_MS,
            syn_e_rev_mV=SIM_SYN_E_REV_MV,
            post_active=SIM_POST_ACTIVE,
            gap_enabled=bool(preset["gap_enabled"]),
            gap_mode=SIM_GAP_MODE,
            gap_rectify_direction=SIM_GAP_RECTIFY_DIRECTION,
            gap_g_uS=SIM_GAP_G_US,
            gap_all_synapses=SIM_GAP_ALL_SYNAPSES,
            gap_max_synapses=SIM_GAP_MAX_SYNAPSES,
            include_psi_gap_pairs=SIM_INCLUDE_PSI_GAP_PAIRS,
        )
    else:
        RUN_DIR = None
        BEAM_DF = run_waveform(
            condition="standing_still",
            output_root=DEFAULT_OUTPUT_ROOT,
            phase2_run=None,
            voltage_ids=SIM_VOLTAGE_IDS,
            duration_ms=1000.0,
            dt_ms=0.5,
            show_voltage_plot=False,
        )
else:
    display(Markdown("Set `RUN_DIGIFLY_ESCAPE = True` and rerun this cell to launch the selected notebook preset."))

## Batch Generate All Paper Conditions

This produces one output folder per condition. It does not run NEURON; simulation-gated conditions in the batch will stay flat unless you pass a Phase 2 run folder through the launcher above.

In [ ]:
RUN_BATCH = False

if RUN_BATCH:
    batch_frames = {}
    for condition in CONDITIONS:
        batch_frames[condition] = run_waveform(
            condition=condition,
            output_root=DEFAULT_OUTPUT_ROOT,
            show_beam_plot=False,
            show_voltage_plot=False,
        )
    display(Markdown(f"Generated {len(batch_frames)} conditions under `{DEFAULT_OUTPUT_ROOT}`."))
else:
    display(Markdown("Set `RUN_BATCH = True` and rerun this cell to generate every beam condition."))